# 08 — Interactive Member Selection (v2: all xmatch sources)

Like notebook 07, but over **every source in `hst_xmatch/master_combined_v2.csv`**
— including HST-only stars far fainter than Gaia. Selections seed
`bp3m-pop-fit-v2` via `member_seed_v2.csv` (keyed by `source_index` +
`gaia_source_id`).

**How to use** — identical to notebook 07:
1. Run all cells. Box select is default; lasso via the toolbar.
2. Panel selections **intersect**; a re-draw **replaces** a panel's region
   (tick **Extend** to OR); the **Clear** button under a panel clears it (double-click only resets the zoom). Membership is
   recomputed in Python from the drawn coordinates.
3. Missing photometry is permissive by default (`STRICT_MISSING = False`).
4. **Save member_seed_v2.csv**, run the final summary cell, then:
   `bp3m-pop-fit-v2 --name FIELD --lvd_key KEY --use_member_seed [...]`

**Note on stability**: `source_index` is the row position in
`master_combined_v2.csv`. Re-running `bp3m-v2` regenerates that file and can
reorder rows — redraw the selection after a v2 rerun.

**Setup** — needs `plotly` + `anywidget` in the kernel env; see notebook 07's
header for the JupyterHub labextension symlink fix.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Leo_I'
SEED_OUT   = 'member_seed_v2.csv'  # written into the field directory
VPD_ZOOM   = 3.0                   # initial VPD half-width (mas/yr)
MIN_PAIR   = 50                    # min joint stars for a CMD/CC panel
STRICT_MISSING = False             # True: drawn panel rejects stars lacking data
GAIA_SOURCE = 'v2'                 # 'v2': Gaia-matched stars take v2 posterior PMs
SIG_PM_MAX  = 1.0                  # only show sources with RMS PM sigma below this
N_DET_MIN   = 3                    # ... and at least this many fitted detections
# (SIG_PM_MAX/N_DET_MIN should match the bp3m-pop-fit-v2 eligibility flags:
#  --max_sigma_free_pm / --min_detections — hidden sources can't be selected,
#  and the pop fit would ignore them anyway)
NB_VERSION = '2026-09-16g'          # printed below and stamped on the summary figure: if you don't see it, reload the notebook from disk
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from pathlib import Path

field_dir = Path(OUTPUT_DIR).expanduser().resolve() / FIELD_NAME
print('field:', field_dir, '| notebook version', NB_VERSION)

In [ ]:
# ── Load: xmatch master catalogue (+ v2 posterior PMs for Gaia stars) ───────
from bp3m.pipeline.run_pop_fit_v2 import _load_catalog

mc = _load_catalog(field_dir, gaia_source=GAIA_SOURCE)

# Gaia photometry + parallax for the Gaia-matched sources, so the same Gaia
# CMD and parallax planes as notebook 07 are available here (v2 posterior
# parallax where a v2 run exists, else the v1 one).
_sa_path = field_dir / 'BP3M_v2_results' / 'stellar_astrometry.csv'
if not _sa_path.exists():
    _sa_path = field_dir / 'BP3M_results' / 'stellar_astrometry.csv'
if _sa_path.exists():
    _sa = pd.read_csv(_sa_path, dtype={'Gaia_id': np.int64},
                      usecols=lambda c: c in ('Gaia_id', 'gmag', 'bp_rp', 'parallax_bp3m'))
    _sa = _sa.rename(columns={'Gaia_id': 'gaia_source_id'}).drop_duplicates('gaia_source_id')
    mc = mc.drop(columns=[c for c in ('gmag', 'bp_rp', 'parallax_bp3m') if c in mc.columns])
    mc = mc.merge(_sa, on='gaia_source_id', how='left')
    print(f'Gaia photometry attached for {int(np.isfinite(mc["gmag"]).sum())} Gaia-matched sources')
else:
    mc['gmag'] = np.nan; mc['bp_rp'] = np.nan; mc['parallax_bp3m'] = np.nan
    print('no stellar_astrometry.csv — Gaia CMD panel unavailable')

# G for every source: Gaia G where matched; for HST-only sources an estimate
# from the reddest HST band with data, G_est = mag + median(G − mag) over the
# Gaia-matched stars in that band ("extended" magnitudes).
_bands_red_first = sorted((c for c in mc.columns if c.startswith('mag_wmean_')),
                          key=lambda c: -float(''.join(ch for ch in c.split('_')[-1] if ch.isdigit()) or 0))
mc['G_any'] = mc['gmag']
for _b in _bands_red_first:
    _ok = np.isfinite(mc['gmag']) & np.isfinite(mc[_b])
    if _ok.sum() < 10:
        continue
    _off = np.nanmedian(mc.loc[_ok, 'gmag'] - mc.loc[_ok, _b])
    _need = ~np.isfinite(mc['G_any']) & np.isfinite(mc[_b])
    mc.loc[_need, 'G_any'] = mc.loc[_need, _b] + _off
# parallax: v2/v1 posterior for Gaia-matched sources, xmatch fit otherwise
mc['plx_any'] = np.where(np.isfinite(mc['parallax_bp3m']), mc['parallax_bp3m'],
                         pd.to_numeric(mc.get('parallax_xmatch'), errors='coerce'))

sig_rms = np.sqrt((mc['sig_ra'] ** 2 + mc['sig_dec'] ** 2) / 2)
n_det_fit = pd.to_numeric(mc.get('n_detect_fit'), errors='coerce').fillna(0)
is_gaia = (mc['gaia_source_id'] != 0).to_numpy()
# Quality gate for HST-only sources only: Gaia-matched stars are ALWAYS shown
# (the bright, saturated ones have 1-2 HST detections but the best Gaia PMs —
# dropping them cost Pal5 its absolute-frame anchors, 2026-09-16).
_qual = (np.isfinite(sig_rms) & (sig_rms > 0) & (sig_rms < SIG_PM_MAX)
         & (n_det_fit >= N_DET_MIN)).to_numpy()
show = np.isfinite(mc['pm_ra']).to_numpy() & np.isfinite(mc['pm_dec']).to_numpy() & (is_gaia | _qual)
master = mc.loc[show].reset_index(drop=True)
# selection id: source_index as string (plotly customdata must avoid
# JS-number roundtrips; gaia ids especially)
master['sid_str'] = master['source_index'].astype(str)
print(f"{show.sum()} of {len(mc)} sources shown: "
      f"{int((master['gaia_source_id'] != 0).sum())} Gaia-matched (all), "
      f"{int((master['gaia_source_id'] == 0).sum())} HST-only "
      f"(sigma_rms < {SIG_PM_MAX}, n_detect_fit >= {N_DET_MIN})")

In [ ]:
# ── Panel definitions ────────────────────────────────────────────────────────
_WAVELENGTHS = {  # nm, blue→red ordering
    'F275W': 275, 'F336W': 336, 'F390W': 390, 'F435W': 435, 'F438W': 438,
    'F475W': 475, 'F555W': 555, 'F606W': 606, 'F625W': 625, 'F775W': 775,
    'F814W': 814, 'F850LP': 900, 'F110W': 1100, 'F125W': 1250, 'F160W': 1600,
}

def _wl(band):
    return _WAVELENGTHS.get(band.split('/')[0].upper(), 9999)

mag_cols = sorted((c for c in master.columns if c.startswith('mag_wmean_')),
                  key=lambda c: _wl(c.replace('mag_wmean_', '')))
bands = [c.replace('mag_wmean_', '') for c in mag_cols]
print('bands:', bands)

panels = []   # (title, x, y, xlabel, ylabel, invert_y)
panels.append(('VPD (xmatch/v2)', master['pm_ra'], master['pm_dec'],
               'pmra [mas/yr]', 'pmdec [mas/yr]', False))

def _n_joint(*cols):
    m = np.ones(len(master), bool)
    for c in cols:
        m &= np.isfinite(master[c].to_numpy(float))
    return int(m.sum())

_skipped = []
for i in range(len(bands)):
    for j in range(i + 1, len(bands)):
        b, r = bands[i], bands[j]
        if b.split('/')[0] == r.split('/')[0]:
            _skipped.append(f'{b} × {r} (same filter)')
            continue
        n = _n_joint(f'mag_wmean_{b}', f'mag_wmean_{r}')
        if n < MIN_PAIR:
            _skipped.append(f'{b} × {r} (n={n})')
            continue
        panels.append((f'{b} − {r} CMD  [n={n}]',
                       master[f'mag_wmean_{b}'] - master[f'mag_wmean_{r}'],
                       master[f'mag_wmean_{r}'],
                       f'{b} − {r}', r, True))

if len(bands) >= 3:
    from itertools import combinations
    for b1, b2, b3 in combinations(bands, 3):
        if len({b.split('/')[0] for b in (b1, b2, b3)}) < 3:
            continue
        n = _n_joint(f'mag_wmean_{b1}', f'mag_wmean_{b2}', f'mag_wmean_{b3}')
        if n < MIN_PAIR:
            _skipped.append(f'{b1} × {b2} × {b3} (n={n})')
            continue
        panels.append((f'({b1}−{b2}) vs ({b2}−{b3})  [n={n}]',
                       master[f'mag_wmean_{b1}'] - master[f'mag_wmean_{b2}'],
                       master[f'mag_wmean_{b2}'] - master[f'mag_wmean_{b3}'],
                       f'{b1} − {b2}', f'{b2} − {b3}', False))

# Gaia CMD — Gaia-matched sources only (HST-only sources have no BP−RP and
# pass through a box drawn here unless STRICT_MISSING)
if {'gmag', 'bp_rp'}.issubset(master.columns) and np.isfinite(master['bp_rp']).sum() >= 10:
    panels.append(('Gaia CMD (Gaia-matched)', master['bp_rp'], master['gmag'],
                   'BP − RP', 'G', True))

# Parallax plane vs G, as in notebook 07 but extended: G is Gaia G for
# Gaia-matched sources and G_est (reddest HST band + median Gaia offset) for
# HST-only ones; parallax is the v2/v1 posterior for Gaia-matched sources and
# the xmatch fit otherwise (often exactly 0 for faint HST-only stars, whose
# parallax is not fitted — draw the box accordingly).
if {'G_any', 'plx_any'}.issubset(master.columns) and _n_joint('G_any', 'plx_any') >= MIN_PAIR:
    panels.append(('Parallax vs G (G_est for HST-only)', master['G_any'],
                   master['plx_any'], 'G / G_est', 'parallax [mas]', False))

if _skipped:
    print(f'skipped {len(_skipped)} sparse/degenerate panels:')
    for s in _skipped:
        print('  ', s)
print(f'{len(panels)} panels: ' + ', '.join(p[0] for p in panels))

In [ ]:
# ── Interactive selection UI ─────────────────────────────────────────────────
import plotly.graph_objects as go
import ipywidgets as W

gid_all       = master['sid_str'].to_numpy()
panel_sel     = {}      # panel index -> set of sid_str (None = no constraint)
panel_missing = {}
panel_shapes  = {}      # panel index -> plotly selection outline(s) to restore after a deselect      # panel index -> sources NOT plotted there
figs          = []

status = W.HTML()
version_lbl = W.HTML(f'<span style="color:#888">notebook {NB_VERSION}</span>')

def combined_selection():
    active = [(k, s) for k, s in panel_sel.items() if s is not None]
    if not active:
        return None
    out = set(gid_all)
    for k, s in active:
        allowed = s if STRICT_MISSING else (s | panel_missing.get(k, set()))
        out &= allowed
    return out

def _n_partially_constrained(comb):
    active = [k for k, s in panel_sel.items() if s is not None]
    return sum(1 for g in comb
               if any(g in panel_missing.get(k, ()) for k in active))

def refresh():
    comb = combined_selection()
    for fw in figs:
        tr = fw.data[0]
        if comb is None:
            tr.selectedpoints = None
        else:
            ids = tr.customdata[:, 0]
            tr.selectedpoints = [k for k, g in enumerate(ids) if g in comb]
    n = len(comb) if comb is not None else len(gid_all)
    n_active = sum(1 for s in panel_sel.values() if s is not None)
    _extra = ''
    if comb is not None and not STRICT_MISSING:
        n_part = _n_partially_constrained(comb)
        if n_part:
            _extra = (f' — <span style="color:#b8860b">{n_part} selected source'
                      f'{"s" if n_part != 1 else ""} missing data in ≥1 drawn panel</span>')
    status.value = (f'<b>{n}</b> sources selected '
                    f'({n_active} panel constraint{"s" if n_active != 1 else ""} active)'
                    f'{_extra}')

extend_chk = W.Checkbox(value=False, description='Extend (OR new regions into a panel)',
                        indent=False, layout=W.Layout(width='320px'))

import re as _re
import asyncio as _asyncio

# ── Selection bookkeeping ────────────────────────────────────────────────────
# Three independent ways a drawn region reaches Python, all feeding the same
# recompute so the result cannot depend on plotly's event quirks:
#  1. on_selection (point event): immediate, but for Scattergl the point list is
#     EMPTY after an autoscale/zoom and the widget then sends nothing at all
#     (Pal5 v2 VPD, 2026-09-16) — so it can't be the only path.
#  2. layout.selections observer: fires on the relayout / layout delta; the
#     layout.selections PROPERTY is stale then, so outlines are read from the
#     raw layout JSON (see _current_outlines), now and again 0.3 s / 1.5 s later.
#  3. _sync_all(): Save and the summary cell re-derive every panel from the
#     outlines currently on screen, whatever events did or did not fire.

def _shape_mask(shape, xs, ys):
    """Mask of plotted points inside one outline; None for a zero-area click."""
    stype = getattr(shape, 'type', None) or shape.get('type')
    if stype == 'path':                                          # lasso
        path = getattr(shape, 'path', None) or shape.get('path')
        from matplotlib.path import Path as _MplPath
        nums = _re.findall(r'[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?', path)
        xy = np.asarray(nums, float).reshape(-1, 2)
        return _MplPath(xy).contains_points(np.column_stack([xs, ys]))
    g = lambda k: float(getattr(shape, k, None) if hasattr(shape, k) else shape[k])
    x0, x1 = sorted((g('x0'), g('x1')))                          # box
    y0, y1 = sorted((g('y0'), g('y1')))
    if x0 == x1 or y0 == y1:
        return None
    return (xs >= x0) & (xs <= x1) & (ys >= y0) & (ys <= y1)

def _shape_key(s):
    g = lambda k: getattr(s, k, None) if hasattr(s, k) else s.get(k)
    vals = [g(k) for k in ('type', 'x0', 'x1', 'y0', 'y1', 'path')]
    return tuple(round(v, 9) if isinstance(v, float) else v for v in vals)

def _apply_shapes(panel_idx, shapes):
    """Recompute a panel's constraint from outlines. False if none has area."""
    tr = figs[panel_idx].data[0]
    xs = np.asarray(tr.x, float); ys = np.asarray(tr.y, float)
    ids = tr.customdata[:, 0]
    mask = np.zeros(len(xs), bool); n_real = 0
    for s in shapes:
        m = _shape_mask(s, xs, ys)
        if m is None:
            continue
        mask |= m; n_real += 1
    if n_real == 0:
        return False
    # An empty outline is a real constraint (nobody passes), never "no constraint".
    panel_sel[panel_idx]    = {ids[k] for k in np.where(mask)[0]}
    panel_shapes[panel_idx] = tuple(shapes)
    return True

def _merge_shapes(panel_idx, shapes):
    """Apply re-draw-replaces / Extend-ORs semantics to incoming outlines."""
    prev = panel_shapes.get(panel_idx, ())
    prev_keys = {_shape_key(p) for p in prev}
    new = [s for s in shapes if _shape_key(s) not in prev_keys]
    if extend_chk.value and prev and new:
        return tuple(prev) + tuple(new)
    if not extend_chk.value and len(shapes) > 1 and new:
        return tuple(new[-1:])
    return tuple(shapes)

def _current_outlines(fw):
    # The layout.selections PROPERTY is stale for outlines drawn in the
    # browser (plotly updates its raw props but not the compound-object
    # cache), so read the raw JSON, which is always current.
    return tuple(fw.layout.to_plotly_json().get('selections') or ())

def _process_panel(panel_idx):
    """Re-derive one panel's constraint from the outlines currently on screen."""
    fw = figs[panel_idx]
    shapes = _current_outlines(fw)
    if not shapes:
        # plotly erased the outline (click, double-click, zoom reset) or the
        # outline has not landed yet: keep the constraint, restore the outline.
        if panel_sel.get(panel_idx) is not None and panel_shapes.get(panel_idx):
            fw.layout.selections = panel_shapes[panel_idx]
    else:
        shapes = _merge_shapes(panel_idx, shapes)
        if _apply_shapes(panel_idx, shapes):
            if [_shape_key(s) for s in shapes] != [_shape_key(s) for s in _current_outlines(fw)]:
                fw.layout.selections = shapes
        else:                                                    # only zero-area clicks
            fw.layout.selections = panel_shapes.get(panel_idx, ())
    refresh()

def _sync_all():
    for k in range(len(figs)):
        _process_panel(k)

def _defer(fn, *args):
    """Run fn after the pending widget messages (layout delta) have landed."""
    try:
        loop = _asyncio.get_running_loop()
    except RuntimeError:
        try:
            loop = _asyncio.get_event_loop()
        except RuntimeError:
            fn(*args); return
    for dt in (0.3, 1.5):
        loop.call_later(dt, fn, *args)

def _make_layout_handler(panel_idx):
    def _on_selections(layout, _stale_value):
        _process_panel(panel_idx)          # raw props are already current here
        _defer(_process_panel, panel_idx)  # and again once any late delta lands
    return _on_selections

def _make_select_handler(panel_idx):
    def _on_select(trace, points, selector):
        # Point event: rebuild the region from the selector geometry (never
        # from points.point_inds) and apply it right away.
        if getattr(selector, 'xrange', None) is not None:
            x0, x1 = selector.xrange; y0, y1 = selector.yrange
            shape = dict(type='rect', x0=x0, x1=x1, y0=y0, y1=y1, xref='x', yref='y')
        elif getattr(selector, 'xs', None) is not None:
            pts = ''.join(f'L{x},{y}' for x, y in zip(selector.xs, selector.ys))
            shape = dict(type='path', path='M' + pts[1:] + 'Z', xref='x', yref='y')
        else:
            return
        shapes = _merge_shapes(panel_idx, (shape,))
        if _apply_shapes(panel_idx, shapes):
            refresh()
        _defer(_process_panel, panel_idx)      # reconcile with plotly's stored outline
    return _on_select

def _make_deselect(panel_idx):
    def _on_deselect(trace, points):
        # click / double-click: constraints persist; only Clear removes them
        _defer(_process_panel, panel_idx)
    return _on_deselect

for k, (title, x, y, xl, yl, inv) in enumerate(panels):
    fin = np.isfinite(np.asarray(x, float)) & np.isfinite(np.asarray(y, float))
    panel_missing[k] = set(gid_all[~fin])
    fw = go.FigureWidget(
        data=[go.Scattergl(
            x=np.asarray(x, float)[fin], y=np.asarray(y, float)[fin],
            mode='markers',
            marker=dict(size=3, color='#1f77b4'),
            customdata=np.c_[gid_all[fin]],
            selected=dict(marker=dict(color='#d62728', size=5)),
            unselected=dict(marker=dict(opacity=0.15)),
            hovertemplate='%{customdata[0]}<extra></extra>',
        )],
        layout=go.Layout(
            title=dict(text=title, font=dict(size=13)),
            width=430, height=380, dragmode='select',
            margin=dict(l=55, r=10, t=40, b=45),
            xaxis=dict(title=xl), yaxis=dict(title=yl),
        ),
    )
    if inv:
        fw.layout.yaxis.autorange = 'reversed'
    if title.startswith('VPD'):
        fw.layout.xaxis.range = [-VPD_ZOOM, VPD_ZOOM]
        fw.layout.yaxis.range = [-VPD_ZOOM, VPD_ZOOM]
    figs.append(fw)
    fw.data[0].on_selection(_make_select_handler(k))
    fw.data[0].on_deselect(_make_deselect(k))
    fw.layout.on_change(_make_layout_handler(k), 'selections')

def _make_clear(panel_idx):
    def _clear_panel(_btn=None):
        panel_sel[panel_idx] = None
        panel_shapes.pop(panel_idx, None)
        figs[panel_idx].layout.selections = ()
        refresh()
    return _clear_panel

panel_clear_btns = []
for k, (title, *_rest) in enumerate(panels):
    _b = W.Button(description=f'Clear: {title[:28]}', layout=W.Layout(width='430px'))
    _b.on_click(_make_clear(k))
    panel_clear_btns.append(_b)

def _save(_btn=None):
    _sync_all()                     # honour every outline on screen, event or not
    comb = combined_selection()
    if comb is None:
        status.value = '<b style="color:red">Nothing selected — draw a box first.</b>'
        return
    sel_mask = master['sid_str'].isin(comb)
    out = master.loc[sel_mask, ['source_index', 'gaia_source_id']].copy()
    out['trusted'] = True
    out = out.sort_values('source_index')
    out_path = field_dir / SEED_OUT
    out.to_csv(out_path, index=False)
    status.value = (f'<b style="color:green">Saved {len(out)} members '
                    f'({int((out.gaia_source_id != 0).sum())} Gaia, '
                    f'{int((out.gaia_source_id == 0).sum())} HST-only) '
                    f'→ {out_path}</b>')

def _clear(_btn=None):
    panel_sel.clear()
    panel_shapes.clear()
    for fw in figs:
        fw.layout.selections = ()
    refresh()

save_btn  = W.Button(description='Save member_seed_v2.csv', button_style='success')
clear_btn = W.Button(description='Clear all selections')
sync_btn  = W.Button(description='Recompute from outlines')
save_btn.on_click(_save)
clear_btn.on_click(_clear)
sync_btn.on_click(lambda _b=None: _sync_all())

refresh()
boxes = [W.VBox([fw, btn]) for fw, btn in zip(figs, panel_clear_btns)]
rows = [W.HBox(boxes[i:i + 2]) for i in range(0, len(boxes), 2)]
W.VBox([W.HBox([save_btn, sync_btn, clear_btn, extend_chk, status, version_lbl]), *rows])

In [ ]:
# ── Final selection summary (run AFTER drawing/saving your selection) ───────
import matplotlib.pyplot as plt

if '_sync_all' in globals():
    _sync_all()                 # pick up outlines whose events never arrived
comb = combined_selection()
if comb is None and (field_dir / SEED_OUT).exists():
    _saved = pd.read_csv(field_dir / SEED_OUT)
    comb = set(_saved['source_index'].astype(str))
    print(f'No live selection — loaded {len(comb)} members from {SEED_OUT}')

if comb is None:
    print('No selection drawn and no saved seed CSV — nothing to summarise.')
else:
    sel_mask = master['sid_str'].isin(comb).to_numpy()
    ncol = 3
    nrow = int(np.ceil(len(panels) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.0 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (title, x, y, xl, yl, inv) in zip(axes, panels):
        xv = np.asarray(x, float); yv = np.asarray(y, float)
        fin = np.isfinite(xv) & np.isfinite(yv)
        ax.scatter(xv[fin & ~sel_mask], yv[fin & ~sel_mask],
                   s=2, c='0.8', lw=0, label='not selected')
        ax.scatter(xv[fin & sel_mask], yv[fin & sel_mask],
                   s=5, c='crimson', lw=0, label='selected member')
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(f'{title}  ({int((fin & sel_mask).sum())} sel)', fontsize=10)
        if inv:
            ax.invert_yaxis()
        # Focus every panel on the selected members (15% margin) so the
        # figure shows the selection rather than the whole field.
        xs = xv[fin & sel_mask]; ys = yv[fin & sel_mask]
        if len(xs) >= 2:
            def _lim(v):
                lo, hi = float(np.nanmin(v)), float(np.nanmax(v))
                pad = 0.15 * max(hi - lo, 1e-3)
                return lo - pad, hi + pad
            ax.set_xlim(*_lim(xs))
            y0, y1 = _lim(ys)
            ax.set_ylim((y1, y0) if inv else (y0, y1))
        elif title.startswith('VPD'):
            ax.set_xlim(-VPD_ZOOM, VPD_ZOOM); ax.set_ylim(-VPD_ZOOM, VPD_ZOOM)
    for ax in axes[len(panels):]:
        ax.set_visible(False)
    axes[0].legend(fontsize=8, loc='upper right')
    fig.suptitle(f'{FIELD_NAME} — v2 member selection: '
                 f'{int(sel_mask.sum())} of {len(master)} shown sources', y=0.995)
    fig.text(0.995, 0.005, f'notebook {NB_VERSION}', ha='right', va='bottom', fontsize=7, color='0.5')
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    out_png = field_dir / 'member_seed_v2_selection.png'
    # Render on an explicit Agg canvas — independent of the notebook backend
    # (inline savefig wrote fully transparent PNGs, 2026-09-16) — and display
    # the saved file itself, so what you see is exactly what was written.
    from matplotlib.backends.backend_agg import FigureCanvasAgg as _Agg
    _Agg(fig)
    fig.savefig(out_png, dpi=150, bbox_inches='tight', facecolor='white', transparent=False)
    plt.close(fig)
    print(f'Saved summary figure → {out_png}')
    from IPython.display import Image as _Img, display as _disp
    _disp(_Img(filename=str(out_png)))

## Using the selection

```bash
# seed only: starting member set; the fit refines membership freely
bp3m-pop-fit-v2 --name FIELD --lvd_key KEY --use_member_seed [...]

# frozen: the fit may REMOVE seed sources but can never add outsiders
bp3m-pop-fit-v2 --name FIELD --lvd_key KEY --use_member_seed --freeze_member_seed [...]
```

Both auto-load `FIELD/member_seed_v2.csv` (fallback `member_seed.csv`);
`--member_seed_csv PATH` overrides. The seed matches on `source_index`
(row of `master_combined_v2.csv`) plus `gaia_source_id` where present.